[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/crystal_waves/crystal_waves.ipynb)

# Elastic Waves in Crystals

A crystal's stiffness is a map with three slots: give it the normal of a face and a displacement that varies along a direction, and it returns the traction on that face. Every question about elastic waves is a way of filling those slots. Fill two with the direction a wave travels and what is left is the Christoffel map on displacements, whose eigenpairs are the wave speeds and polarizations. Fill them with the polarization and the direction and what is left is a vector: where the wave's energy goes. In a crystal that is not along the wave, and where the energy of many directions converges, heat pulses focus into caustics.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from numga import NumpyContext
from numga.algebras import VGA3D
from examples.mechanics.crystal_waves import render

np.set_printoptions(precision=3, suppress=True)

ga = VGA3D                                                # the geometric algebra of the crystal's directions
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
Stiffness = ga.gatype((Vector, Vector, Vector, Vector))   # traction <- (normal, displacement, gradient)
axes = mv.basis()                                         # [3] Vector: the crystal's axes x, y, z
x, y, z = axes

# Silicon and the far more anisotropic beta-brass: cubic elastic constants in GPa and densities in
# g/cm^3, so that speeds come out in km/s.
names = ["silicon", "beta-brass"]
c11 = np.array([165.7, 129.1])
c12 = np.array([63.9, 109.7])
c44 = np.array([79.6, 82.4])
density = np.array([2.33, 7.60])
print(Stiffness)

## 1. The stiffness as a map

For an isotropic solid the traction has two terms. The dilation `v | h` pushes along the normal, and the shear pulls along the displacement and its gradient. The shear needs the identity `n | (v ^ h) == (n | v) * h - (n | h) * v`, so that both of its terms are written with the slots in the same order. A cubic crystal adds one term, in which each of its axes reads all three slots: it is how far the crystal is from isotropic, `c11 - c12 - 2 * c44`. Writing the open slots as the type `Vector` builds the map; each term below is one product with three slots open.

In index notation the stiffness reads as the fourth-rank tensor $C_{ijkl}$, the traction on a face of normal $n$ as $t_i = C_{ijkl}\, n_j\, \partial_l u_k$, and the cubic stiffness as $C_{ijkl} = c_{12}\,\delta_{ij}\delta_{kl} + c_{44}(\delta_{ik}\delta_{jl} + \delta_{il}\delta_{jk}) + (c_{11} - c_{12} - 2c_{44}) \sum_a e^a_i e^a_j e^a_k e^a_l$.

In [ ]:
def stiffness(c11: np.ndarray, c12: np.ndarray, c44: np.ndarray) -> Stiffness:
    # The slots in order: the face's normal, the displacement, its gradient. The dilation is the
    # normal times the displacement's inner product with the gradient; the shear, the displacement
    # times the normal's inner product with the gradient plus the gradient times the normal's with
    # the displacement; the cubic term, each axis reading all three slots.
    dilation = Vector * (Vector | Vector)                                        # [] Vector <- (Vector, Vector, Vector)
    shear = 2 * (Vector | Vector) * Vector - (Vector | (Vector ^ Vector))        # [] Vector <- (Vector, Vector, Vector)
    cubic = (axes * (axes | Vector) * (axes | Vector) * (axes | Vector)).sum()   # [] Vector <- (Vector, Vector, Vector)
    # The constants as scalars, which carry their batch into the stiffness: one per material.
    c11, c12, c44 = (mv.scalar(np.asarray(value, dtype=float)[..., None]) for value in (c11, c12, c44))
    return c12 * dilation + c44 * shear + (c11 - c12 - 2 * c44) * cubic


crystals = stiffness(c11, c12, c44)                                 # [materials] Vector <- (Vector, Vector, Vector)
# The elastic constants read back as tractions: a stretch along x pulls on the x face with c11 and
# on the y face with c12; a shear of the xy face pulls on it with c44.
read_back = [x | crystals(x, x, x), y | crystals(y, x, x), x | crystals(y, x, y)]   # [materials] Scalar each

In [ ]:
print(crystals)
print("c11, c12, c44 read back:", [value.to_array() for value in read_back])

## 2. Waves along a heading

A plane wave displaces the material by its polarization, varying along its heading; the traction it produces on the wavefront, whose normal is the heading again, must accelerate the material. So the heading goes into the normal and gradient slots, and what is left is the Christoffel map on displacements. It is symmetric, and its eigenpairs are density times squared speed and the polarization of each of the three waves: one compressional, two shear.

In index notation this reads as the Christoffel equation, $\Gamma_{ik} = C_{ijkl}\, n_j n_l$ with $\det(\Gamma - \rho v^2 \delta) = 0$.

In [ ]:
headings = mv.vector(([[1, 0, 0], [1, 1, 0], [1, 1, 1]])).normalized()   # [3] Vector: [100], [110], [111]
christoffel = crystals[:, None](headings, Vector, headings)                       # [materials, 3] Vector <- Vector
values, polarizations = christoffel.eigh()                                        # [materials, 3, 3] Scalar, Vector
speeds = (values / density[:, None, None]).square_root()                          # [materials, 3, 3] Scalar: km/s, slowest first
# Along a cube edge the textbook speeds are sqrt(c44 / rho) twice and sqrt(c11 / rho).
textbook_edge = np.sqrt(np.stack([c44, c44, c11], axis=-1) / density[:, None])   # [materials, 3]

In [ ]:
print(christoffel)
print("speeds along [100], [110], [111]:\n", speeds.to_array())
print("textbook along [100]:", textbook_edge)

## 3. Where the energy goes

The energy a wave carries flows with the traction it exerts times the velocity of the material. Both are along the polarization, so the polarization goes into the normal and displacement slots and the heading into the gradient slot. Nothing is left open: the result is a vector, the energy flux. Divided by density times speed it is the group velocity.

In index notation the group velocity reads as $v^g_i = C_{ijkl}\, p_j p_k n_l / \rho v$. Its component along the heading is the phase speed, but in a crystal it also has a component across it, and many headings can send their energy the same way. A heat pulse from a point then arrives on a crystal face along bright lines. Below, evenly spread headings are binned by where their energy lands on the face z = 1.

In [ ]:
def energy_flow(crystal: Stiffness, heading, polarization, density: np.ndarray):
    # The crystal, its density and the heading against each of the three waves.
    crystal, density, along = crystal[..., None], density[..., None], heading[..., None]
    squared = polarization | crystal(along, polarization, along)        # [..., 3] density times squared speed
    return crystal(polarization, polarization, along) / (squared * density).square_root()


rng = np.random.default_rng(0)
spread = mv.vector(rng.normal(size=(200_000, 3))).normalized()          # [n] Vector: headings over the sphere
# Both crystals at once, against every heading.
_, polarization = crystals[:, None](spread, Vector, spread).eigh()      # [materials, n, 3] Vector
velocities = energy_flow(crystals[:, None], spread, polarization, density[:, None])   # [materials, n, 3] Vector
render.draw_focusing(velocities, names, 300);

The bright lines are caustics: folds in the map from heading to energy direction. Silicon's shear waves make a box and a cross; beta-brass, far more anisotropic, folds them into loops and petals. The compressional wave, nearly isotropic, barely focuses. A section through the wave surfaces, the group velocities for headings in one cube face, shows the folds as cusps.

In [ ]:
angle = np.linspace(0.0, 2 * np.pi, 3000, endpoint=False)
# The x axis turned through every angle in the face z = 0.
around = (mv.xy * (-angle / 2)).exp() >> mv.x                            # [n] Vector
_, polarization = crystals[:, None](around, Vector, around).eigh()      # [materials, n, 3] Vector
fronts = energy_flow(crystals[:, None], around, polarization, density[:, None])   # [materials, n, 3] Vector
render.draw_wave_fronts(fronts, names);

In [ ]:
# checks
np.testing.assert_allclose([value.to_array() for value in read_back], [c11, c12, c44], rtol=1e-10)
np.testing.assert_allclose(speeds[:, 0].to_array(), textbook_edge, rtol=1e-8)
# Along its heading the energy moves at the phase speed.
flow = energy_flow(crystals[:, None], headings, polarizations, density[:, None])
np.testing.assert_allclose((flow | headings[:, None]).to_array(), speeds.to_array(), rtol=1e-8)